# Part 1 — Plain CNN + data augmentation

**Previous result** (`plain_cnn`, no augmentation): train acc ~100% after ~12 epochs, best val acc **20.7%**, val loss rising from 4.15 to 14.4 → the network memorizes the 33 images per class.

**Change tested here:** data augmentation (random resized crop + horizontal flip), so the network never sees exactly the same image twice. Everything else is identical. Set `augment=False` to reproduce the baseline.

## Setup

In [ ]:
import os, json, time, math, random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from torchvision.datasets import FGVCAircraft
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt

try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# If the dataset download gets interrupted, delete DATA_ROOT and re-run:
# torchvision treats a half-extracted folder as a complete dataset.
DATA_ROOT = Path("/content/data") if IN_COLAB else Path.home() / "data" / "fgvc"
RESULTS_DIR = Path("/content/drive/MyDrive/ipcv/results") if IN_COLAB else Path("results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
NUM_WORKERS = os.cpu_count() if IN_COLAB else 0

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print("device:", DEVICE, "| workers:", NUM_WORKERS)

## Config

In [ ]:
config = dict(
    name="plain_cnn_aug",
    seed=42,
    # data
    img_size=128,
    augment=True,
    crop_scale=(0.5, 1.0),   # RandomResizedCrop keeps 50-100% of the image area
    # model
    channels=(32, 64, 128, 256),
    # training (unchanged from the baseline)
    lr=1e-3,
    batch_size=64,
    epochs=30,
)

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

set_seed(config["seed"])

## Data

- **Banner crop:** every FGVC image has a 20px copyright banner at the bottom ("Photo by … / Airliners.net"); we remove it so the network can't use it.
- **Resize** the whole image to `img_size × img_size`, as in the baseline.
- **Augmentation (train only):** `RandomResizedCrop` on the resized image + `RandomHorizontalFlip`. Cropping *after* resizing keeps object proportions the same at train and test time (the train/test discrepancy discussed in FixRes, lecture 6). A horizontal flip doesn't change the aircraft model.
- **Normalization:** per-channel mean/std measured on the train split.

In [ ]:
MEAN, STD = (0.488, 0.519, 0.544), (0.211, 0.203, 0.236)   # computed on the train split

def remove_banner(img):
    return img.crop((0, 0, img.width, img.height - 20))

def make_transform(cfg, train):
    t = [remove_banner, T.Resize((cfg["img_size"], cfg["img_size"]))]
    if train and cfg["augment"]:
        t += [T.RandomResizedCrop(cfg["img_size"], scale=cfg["crop_scale"]), T.RandomHorizontalFlip()]
    t += [T.ToTensor(), T.Normalize(MEAN, STD)]
    return T.Compose(t)

def make_dataset(split, cfg, train=False):
    return FGVCAircraft(DATA_ROOT, split=split, annotation_level="variant", download=True,
                        transform=make_transform(cfg, train))

train_ds = make_dataset("train", config, train=True)
train_plain_ds = make_dataset("train", config)   # same images, no augmentation
val_ds = make_dataset("val", config)
CLASSES = train_ds.classes
NUM_CLASSES = len(CLASSES)
print(f"classes: {NUM_CLASSES} | train {len(train_ds)} | val {len(val_ds)}")

What the network actually sees: one training image, augmented 6 times.

In [ ]:
def show(t):
    return (t.permute(1, 2, 0) * torch.tensor(STD) + torch.tensor(MEAN)).clamp(0, 1)

fig, axes = plt.subplots(1, 7, figsize=(18, 3))
axes[0].imshow(show(train_plain_ds[0][0]))
axes[0].set_title("no augmentation")
for ax in axes[1:]:
    ax.imshow(show(train_ds[0][0])); ax.set_title("augmented")
for ax in axes:
    ax.axis("off")
fig.suptitle(CLASSES[train_ds[0][1]]); plt.tight_layout(); plt.show()

## Model (unchanged): plain CNN

In [ ]:
class PlainCNN(nn.Module):
    def __init__(self, channels, num_classes, img_size):
        super().__init__()
        layers, c_in = [], 3
        for c_out in channels:
            layers += [nn.Conv2d(c_in, c_out, kernel_size=3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2)]
            c_in = c_out
        self.features = nn.Sequential(*layers)
        spatial = img_size // 2 ** len(channels)
        self.classifier = nn.Linear(c_in * spatial * spatial, num_classes)

    def forward(self, x):
        return self.classifier(torch.flatten(self.features(x), 1))

def build_model(cfg):
    return PlainCNN(cfg["channels"], NUM_CLASSES, cfg["img_size"]).to(DEVICE)

print(f"parameters: {sum(p.numel() for p in build_model(config).parameters()):,}")

## Training loop

In [ ]:
def run_epoch(model, loader, opt=None):
    """One pass over loader; trains if an optimizer is given, otherwise evaluates."""
    model.train(opt is not None)
    loss_sum, correct, n = 0.0, 0, 0
    with torch.set_grad_enabled(opt is not None):
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            logits = model(x)
            loss = F.cross_entropy(logits, y)
            if opt is not None:
                opt.zero_grad()
                loss.backward()
                opt.step()
            loss_sum += loss.item() * len(y)
            correct += (logits.argmax(1) == y).sum().item()
            n += len(y)
    return loss_sum / n, correct / n

def fit(cfg, train_ds, val_ds, verbose=True):
    set_seed(cfg["seed"])
    train_dl = DataLoader(train_ds, cfg["batch_size"], shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
    val_dl = DataLoader(val_ds, 256, num_workers=NUM_WORKERS, pin_memory=True)
    model = build_model(cfg)
    opt = torch.optim.Adam(model.parameters(), lr=cfg["lr"])
    hist = {k: [] for k in ("train_loss", "train_acc", "val_loss", "val_acc")}
    best_acc, best_state = -1.0, None
    for epoch in range(1, cfg["epochs"] + 1):
        t0 = time.time()
        tl, ta = run_epoch(model, train_dl, opt)
        vl, va = run_epoch(model, val_dl)
        for k, v in zip(hist, (tl, ta, vl, va)):
            hist[k].append(v)
        if va > best_acc:  # early stopping: keep the best-on-val weights
            best_acc, best_state = va, {k: v.clone() for k, v in model.state_dict().items()}
        if verbose:
            print(f"ep {epoch:3d} | train loss {tl:.3f} acc {ta:.3f} | val loss {vl:.3f} acc {va:.3f} | {time.time() - t0:.0f}s")
    model.load_state_dict(best_state)
    return model, hist

## Sanity checks

1. Initial loss ≈ ln(100) ≈ 4.61.
2. The model can memorize 100 (non-augmented) images.

In [ ]:
set_seed(config["seed"])
init_loss, _ = run_epoch(build_model(config), DataLoader(Subset(val_ds, range(500)), 256))
print(f"initial loss {init_loss:.3f} (expected ~{math.log(NUM_CLASSES):.3f})")

tiny = Subset(train_plain_ds, range(100))
_, h = fit({**config, "epochs": 60, "batch_size": 20}, tiny, tiny, verbose=False)
print(f"tiny subset: train acc {h['train_acc'][-1]:.2f} (expected ~1.00)")

## Train

In [ ]:
t0 = time.time()
model, hist = fit(config, train_ds, val_ds)
print(f"total {time.time() - t0:.0f}s | best val acc {max(hist['val_acc']):.3f} (epoch {int(np.argmax(hist['val_acc'])) + 1})")

run = dict(config=config, history=hist, best_val_acc=max(hist["val_acc"]))
out = RESULTS_DIR / f"{config['name']}_seed{config['seed']}.json"
out.write_text(json.dumps(run, indent=2))
print("saved", out)

## Comparison with previous runs

Every run saved in `RESULTS_DIR` (put `plain_cnn_seed42.json` there too).

In [ ]:
runs = {p.stem: json.loads(p.read_text()) for p in sorted(RESULTS_DIR.glob("*.json"))}

fig, axes = plt.subplots(1, 3, figsize=(17, 4))
for name, r in runs.items():
    h = r["history"]
    ep = range(1, len(h["val_acc"]) + 1)
    line, = axes[0].plot(ep, h["train_acc"], "--")
    axes[0].plot(ep, h["val_acc"], color=line.get_color(), label=name)
    axes[1].plot(ep, h["val_loss"], label=name)
    axes[2].plot(ep, h["train_loss"], label=name)
axes[0].set_title("accuracy (dashed = train, solid = val)")
axes[1].set_title("val loss"); axes[2].set_title("train loss")
for ax in axes:
    ax.set_xlabel("epoch"); ax.legend()
plt.tight_layout(); plt.show()

print(f"{'run':30s} {'best val acc':>12s}")
for name, r in runs.items():
    print(f"{name:30s} {r['best_val_acc']:12.3f}")

## Test (final models only)

In [ ]:
RUN_TEST = False
if RUN_TEST:
    test_ds = make_dataset("test", config)
    _, test_acc = run_epoch(model, DataLoader(test_ds, 256, num_workers=NUM_WORKERS))
    print(f"test acc {test_acc:.3f}")